 gcc -o query query.c -I"C:\Program Files\MySQL\MySQL Server 9.0\include" "C:\Program Files\MySQL\MySQL Server 9.0\lib\libmysql.dll"

In [84]:
import os
import mysql.connector

user = os.getenv("MYSQL_USER")
password = os.getenv("MYSQL_PASSWORD")
host = os.getenv("MYSQL_HOST")
database = os.getenv("MYSQL_DATABASE")

connection = mysql.connector.connect(
    host=host,
    user=user,
    password=password,
    database=database
)

cursor = connection.cursor()


In [126]:
import time

# Unoptimized SQL Query
unoptimized_query = """
SELECT DISTINCT CarID, CarType
FROM CombinedBookingsView
WHERE CarID IN (
    SELECT CarID
    FROM (
        SELECT CarID, MONTH(BookingDate) AS BookingMonth
        FROM CombinedBookingsView
        WHERE YEAR(BookingDate) = 2023
    ) AS MonthlyBookings
    GROUP BY CarID
    HAVING COUNT(DISTINCT BookingMonth) = 12
);
"""

# Measure execution time
start_time = time.time()
cursor = connection.cursor()
cursor.execute(unoptimized_query)
result = cursor.fetchall()
end_time = time.time()

# Display results and time
for row in result:
    print(f"CarID: {row[0]}, CarType: {row[1]}")
print(f"Unoptimized Execution Time: {end_time - start_time} seconds")

# Close the connection
cursor.close()


CarID: 6, CarType: Innova
Unoptimized Execution Time: 0.00399470329284668 seconds


True

In [127]:
cursor = connection.cursor()

# Optimized SQL Query Without CTEs
optimized_query = """
SELECT CarID, CarType
FROM (
    SELECT CarID, CarType, COUNT(DISTINCT MONTH(BookingDate)) AS MonthCount
    FROM CombinedBookingsView
    WHERE YEAR(BookingDate) = 2023
      AND CarID IS NOT NULL
    GROUP BY CarID, CarType
) AS CarMonthCounts
WHERE MonthCount = 12;
"""

# Execute the query
start_time = time.time()
cursor.execute(optimized_query)
results = cursor.fetchall()
end_time = time.time()

# Print results and execution time
for row in results:
    print(f"CarID: {row[0]}, CarType: {row[1]}")

print(f"Optimized Execution Time: {end_time - start_time} seconds")

# Close the connection
cursor.close()


CarID: 6, CarType: Innova
Optimized Execution Time: 0.0020580291748046875 seconds


True

# Query 2

In [130]:
cursor = connection.cursor()

# Unoptimized SQL Query
unoptimized_query = """
SELECT CarID, CarType
FROM CombinedBookingsView
WHERE CarID IN (
    SELECT CarID
    FROM CombinedBookingsView
    WHERE YEAR(BookingDate) = 2023
    GROUP BY CarID, CarType
    HAVING COUNT(DISTINCT MONTH(BookingDate)) = 12
)
AND FlightID IS NOT NULL;
"""

# Measure the execution time of the unoptimized query
start_time = time.time()
cursor.execute(unoptimized_query)
unoptimized_result = cursor.fetchall()
end_time = time.time()

print("Unoptimized Query Results:", unoptimized_result)
print("Execution time (unoptimized):", end_time - start_time, "seconds")

cursor.close()


Unoptimized Query Results: [(6, 'Innova')]
Execution time (unoptimized): 0.0032744407653808594 seconds


True

In [131]:
cursor = connection.cursor()

# Optimized SQL Query Without CTEs
optimized_query = """
SELECT DISTINCT cbv.CarID, cbv.CarType
FROM CombinedBookingsView AS cbv
JOIN (
    SELECT CarID, CarType
    FROM CombinedBookingsView
    WHERE YEAR(BookingDate) = 2023
      AND CarID IS NOT NULL
    GROUP BY CarID, CarType
    HAVING COUNT(DISTINCT MONTH(BookingDate)) = 12
) AS AllMonthsCars ON cbv.CarID = AllMonthsCars.CarID AND cbv.CarType = AllMonthsCars.CarType
WHERE cbv.FlightID IS NOT NULL;

"""

# Measure the execution time of the optimized query
start_time = time.time()
cursor.execute(optimized_query)
optimized_result = cursor.fetchall()
end_time = time.time()

# Print the results and execution time
print("Optimized Query Results:")
for row in optimized_result:
    print(f"CarID: {row[0]}, CarType: {row[1]}")
print(f"Execution time (optimized): {end_time - start_time} seconds")

# Close the cursor
cursor.close()


Optimized Query Results:
CarID: 6, CarType: Innova
Execution time (optimized): 0.003011941909790039 seconds


True

# Query 3

In [139]:
cursor = connection.cursor()

unoptimized_query_3 = """
SELECT h.HotelID, h.HotelName, h.City, h.PricePerNight
FROM Hotel AS h
LEFT JOIN Artwork AS a ON h.HotelID = a.hotel_id
WHERE a.hotel_id IS NULL;
"""

# Measure execution time of unoptimized query
start_time = time.time()
cursor.execute(unoptimized_query_3)
unoptimized_result_3 = cursor.fetchall()
end_time = time.time()

print("Unoptimized Query 3 Results:", unoptimized_result_3)
print("Execution time (unoptimized):", end_time - start_time, "seconds")

cursor.close()


Unoptimized Query 3 Results: [(3, 'Ritz', 'Tokyo', Decimal('300.00')), (5, 'Holiday Inn', 'Sydney', Decimal('150.00'))]
Execution time (unoptimized): 0.0020318031311035156 seconds


True

In [140]:
cursor = connection.cursor()

optimized_query_3 = """
SELECT h.HotelID, h.HotelName, h.City, h.PricePerNight
FROM (
    SELECT HotelID, HotelName, City, PricePerNight
    FROM Hotel
) AS h
LEFT JOIN (
    SELECT hotel_id FROM Artwork
) AS a ON h.HotelID = a.hotel_id
WHERE a.hotel_id IS NULL;
"""

# Measure execution time of optimized query
start_time = time.time()
cursor.execute(optimized_query_3)
optimized_result_3 = cursor.fetchall()
end_time = time.time()

print("Optimized Query 3 Results:", optimized_result_3)
print("Execution time (optimized):", end_time - start_time, "seconds")

# Close the connection
cursor.close()


Optimized Query 3 Results: [(3, 'Ritz', 'Tokyo', Decimal('300.00')), (5, 'Holiday Inn', 'Sydney', Decimal('150.00'))]
Execution time (optimized): 0.002581357955932617 seconds


True

# Query 4

In [142]:
cursor = connection.cursor()

# Unoptimized SQL Query
unoptimized_query = """
SELECT DISTINCT u.UserID
FROM User AS u
JOIN CombinedBookingsView AS cbv1 ON u.UserID = cbv1.UserID
JOIN CombinedBookingsView AS cbv2 ON u.UserID = cbv2.UserID
JOIN CombinedBookingsView AS cbv3 ON u.UserID = cbv3.UserID
JOIN Hotel AS h ON cbv3.HotelID = h.HotelID
WHERE
    YEAR(cbv1.BookingDate) = 2022
    AND cbv2.CarType = 'Innova'
    AND h.RoomType = 'Single Room';
"""

# Measure the execution time of the unoptimized query
start_time = time.time()
cursor.execute(unoptimized_query)
unoptimized_result = cursor.fetchall()
end_time = time.time()

# Print the result and execution time
print("Unoptimized Guest List (UserIDs) who made 'Single Room' + 'Innova' booking in 2022:")
for row in unoptimized_result:
    print(f"UserID: {row[0]}")
print("Execution time (unoptimized):", end_time - start_time, "seconds")

cursor.close()


Unoptimized Guest List (UserIDs) who made 'Single Room' + 'Innova' booking in 2022:
UserID: 3
Execution time (unoptimized): 0.03205728530883789 seconds


True

In [75]:
cursor = connection.cursor()

# Simplified Optimized SQL Query
optimized_query = """
SELECT DISTINCT cbv.UserID
FROM CombinedBookingsView AS cbv
JOIN Hotel AS h ON cbv.HotelID = h.HotelID
WHERE
    YEAR(cbv.BookingDate) = 2022
    AND cbv.CarType = 'Innova'
    AND h.RoomType = 'Single Room';
"""

# Measure the execution time of the optimized query
start_time = time.time()
cursor.execute(optimized_query)
optimized_result = cursor.fetchall()
end_time = time.time()

# Print the result and execution time
print("Optimized Guest List (UserIDs) who made 'Single Room' + 'Innova' booking in 2022:")
for row in optimized_result:
    print(f"UserID: {row[0]}")
print("Execution time (optimized):", end_time - start_time, "seconds")

cursor.close()


Optimized Guest List (UserIDs) who made 'Single Room' + 'Innova' booking in 2022:
UserID: 3
Execution time (optimized): 0.0020155906677246094 seconds


True

# Query 5

In [90]:
cursor = connection.cursor()

# Unoptimized SQL Query to get guest names and profile info
unoptimized_query_with_profile = """
SELECT DISTINCT u.UserID, u.Name AS GuestName, u.Email, u.ContactNumber
FROM User AS u
JOIN CombinedBookingsView AS cbv1 ON u.UserID = cbv1.UserID
JOIN CombinedBookingsView AS cbv2 ON u.UserID = cbv2.UserID
JOIN CombinedBookingsView AS cbv3 ON u.UserID = cbv3.UserID
JOIN Hotel AS h ON cbv3.HotelID = h.HotelID
WHERE
    YEAR(cbv1.BookingDate) = 2022
    AND cbv2.CarType = 'Innova'
    AND h.RoomType = 'Single Room';
"""

# Measure the execution time of the unoptimized query
start_time = time.time()
cursor.execute(unoptimized_query_with_profile)
unoptimized_profile_result = cursor.fetchall()
end_time = time.time()

# Print the result and execution time
print("Unoptimized Guest List with Profile Info (UserID, Name, Email, ContactNumber):")
for row in unoptimized_profile_result:
    print(f"UserID: {row[0]}, GuestName: {row[1]}, Email: {row[2]}, ContactNumber: {row[3]}")
print("Execution time (unoptimized):", end_time - start_time, "seconds")

cursor.close()


Unoptimized Guest List with Profile Info (UserID, Name, Email, ContactNumber):
UserID: 3, GuestName: Alice Brown, Email: alice@example.com, ContactNumber: 1230984567
Execution time (unoptimized): 0.03412032127380371 seconds


True

In [91]:
cursor = connection.cursor()

# Optimized SQL Query to get guest names and profile info
optimized_query_with_profile = """
SELECT DISTINCT u.UserID, u.Name AS GuestName, u.Email, u.ContactNumber
FROM CombinedBookingsView AS cbv
JOIN Hotel AS h ON cbv.HotelID = h.HotelID
JOIN User AS u ON cbv.UserID = u.UserID
WHERE
    YEAR(cbv.BookingDate) = 2022
    AND cbv.CarType = 'Innova'
    AND h.RoomType = 'Single Room';
"""

# Measure the execution time of the optimized query
start_time = time.time()
cursor.execute(optimized_query_with_profile)
optimized_profile_result = cursor.fetchall()
end_time = time.time()

# Print the result and execution time
print("Optimized Guest List with Profile Info (UserID, Name, Email, ContactNumber):")
for row in optimized_profile_result:
    print(f"UserID: {row[0]}, GuestName: {row[1]}, Email: {row[2]}, ContactNumber: {row[3]}")
print("Execution time (optimized):", end_time - start_time, "seconds")

cursor.close()


Optimized Guest List with Profile Info (UserID, Name, Email, ContactNumber):
UserID: 3, GuestName: Alice Brown, Email: alice@example.com, ContactNumber: 1230984567
Execution time (optimized): 0.0025217533111572266 seconds


True

# Query 6

In [88]:
cursor = connection.cursor()

# Unoptimized SQL Query
unoptimized_query = """
SELECT u.UserID, u.Name AS GuestName, u.Email, u.ContactNumber
FROM User AS u
LEFT JOIN Booking AS b ON u.UserID = b.UserID
WHERE b.UserID IS NULL;
"""

# Measure the execution time of the unoptimized query
start_time = time.time()
cursor.execute(unoptimized_query)
unoptimized_result = cursor.fetchall()
end_time = time.time()

# Print the result and execution time
print("Unoptimized Query - Users with No Bookings:")
for row in unoptimized_result:
    print(f"UserID: {row[0]}, GuestName: {row[1]}, Email: {row[2]}, ContactNumber: {row[3]}")
print("Execution time (unoptimized):", end_time - start_time, "seconds")

cursor.close()


Unoptimized Query - Users with No Bookings:
UserID: 5, GuestName: Charlie Green, Email: charlie@example.com, ContactNumber: 1234567899
Execution time (unoptimized): 0.0019049644470214844 seconds


True

In [89]:
cursor = connection.cursor()

# Optimized SQL Query
optimized_query = """
SELECT u.UserID, u.Name AS GuestName, u.Email, u.ContactNumber
FROM User AS u
WHERE NOT EXISTS (
    SELECT 1
    FROM Booking AS b
    WHERE b.UserID = u.UserID
);
"""

# Measure the execution time of the optimized query
start_time = time.time()
cursor.execute(optimized_query)
optimized_result = cursor.fetchall()
end_time = time.time()

# Print the result and execution time
print("Optimized Query - Users with No Bookings:")
for row in optimized_result:
    print(f"UserID: {row[0]}, GuestName: {row[1]}, Email: {row[2]}, ContactNumber: {row[3]}")
print("Execution time (optimized):", end_time - start_time, "seconds")

cursor.close()


Optimized Query - Users with No Bookings:
UserID: 5, GuestName: Charlie Green, Email: charlie@example.com, ContactNumber: 1234567899
Execution time (optimized): 0.0015134811401367188 seconds


True